In [ ]:
!pip install numpy==1.26.4 transformers --upgrade

In [ ]:
!pip install torch torchvision --upgrade

In [2]:
!pip uninstall torch torchvision torchaudio -y

Found existing installation: torchvision 0.25.0
Uninstalling torchvision-0.25.0:
  Successfully uninstalled torchvision-0.25.0


You can safely remove it manually.


In [3]:
!pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128

Looking in indexes: https://download.pytorch.org/whl/nightly/cu128
   ---------------------------------------- 0.0/2.8 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 GB 5.1 MB/s eta 0:09:11
   ---------------------------------------- 0.0/2.8 GB 11.5 MB/s eta 0:04:05
   ---------------------------------------- 0.0/2.8 GB 14.1 MB/s eta 0:03:19
   ---------------------------------------- 0.0/2.8 GB 14.0 MB/s eta 0:03:21
   ---------------------------------------- 0.0/2.8 GB 13.3 MB/s eta 0:03:31
   ---------------------------------------- 0.0/2.8 GB 14.0 MB/s eta 0:03:21
   ---------------------------------------- 0.0/2.8 GB 14.1 MB/s eta 0:03:18
   ---------------------------------------- 0.0/2.8 GB 14.2 MB/s eta 0:03:17
   ---------------------------------------- 0.0/2.8 GB 15.1 MB/s eta 0:03:05
   -------------------------------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\ddonz\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [5]:
!py -c "import torch; print(torch.__version__); print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0))"

2.12.0.dev20260306+cu128
True
NVIDIA GeForce RTX 5060 Ti


In [6]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from sklearn.metrics import f1_score
import numpy as np

# Caminho do dataset
pasta_diretorio = r'C:\Users\ddonz\OneDrive\Documentos\Aislan\data'
split_dir = os.path.join(pasta_diretorio, 'split')

excluded_ids = [
    82723, 82687, 82569, 82570, 82576, 82577, 82581, 82587, 82589,
    82624, 82627, 82628, 82642, 82652, 82664, 82665, 82674, 82677,
    82681, 82690, 82705, 82708, 82709, 82738, 82758, 82768,
    82777, 82783, 82784, 82794, 82807, 82812, 82813, 82814,
    82815, 82817, 82819, 82820, 82832, 82845, 82861, 82866,
    82875, 82879, 82895, 82899, 82910, 82912, 82919, 82555,
    82786, 82827, 82927, 82928, 82956, 82968, 83008, 83011,
    83045, 83080, 83086
]
excluded_ids = set(str(x) for x in excluded_ids)

def load_split_filtered(filepath):
    texts, labels = [], []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f.readlines():
            parts = line.strip().split(',', 2)
            participant_id = parts[0].split('/')[1]
            if participant_id not in excluded_ids:
                labels.append(int(parts[1]))
                texts.append(parts[2] if len(parts) > 2 else '')
    return texts, labels

train_texts, train_labels = load_split_filtered(os.path.join(split_dir, 'train.txt'))
val_texts, val_labels = load_split_filtered(os.path.join(split_dir, 'val.txt'))
test_texts, test_labels = load_split_filtered(os.path.join(split_dir, 'test.txt'))

print(f"Train: {len(train_texts)} | Val: {len(val_texts)} | Test: {len(test_texts)}")

C:\Users\ddonz\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train: 598 | Val: 107 | Test: 427


In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando: {device}')

Usando: cuda


In [8]:
print(torch.cuda.is_available())       # True se a GPU está ativa
print(torch.cuda.get_device_name(0))   # Nome da GPU

True
NVIDIA GeForce RTX 5060 Ti


In [9]:
# Dataset
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

class BAHTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(self.labels[idx], dtype=torch.float)
        }

train_dataset = BAHTextDataset(train_texts, train_labels, tokenizer)
val_dataset = BAHTextDataset(val_texts, val_labels, tokenizer)
test_dataset = BAHTextDataset(test_texts, test_labels, tokenizer)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Verificar
batch = next(iter(train_loader))
print(f"input_ids shape: {batch['input_ids'].shape}")
print(f"attention_mask shape: {batch['attention_mask'].shape}")
print(f"labels shape: {batch['label'].shape}")
print(f"labels exemplo: {batch['label'][:5]}")

C:\Users\ddonz\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ddonz\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


input_ids shape: torch.Size([16, 128])
attention_mask shape: torch.Size([16, 128])
labels shape: torch.Size([16])
labels exemplo: tensor([0., 0., 1., 1., 0.])


In [10]:
class BAHTextModel(nn.Module):
    def __init__(self, projection_dim=128, freeze_bert=True):
        super().__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')

        if freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False

        # Projeção: h_t (768) → h_t' (128)
        self.projection = nn.Linear(768, projection_dim)

        # MLP classificador
        self.classifier = nn.Sequential(
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(projection_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, input_ids, attention_mask):
        # BERT → [CLS] token
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        h_t = outputs.pooler_output  # (batch, 768)

        # Projeção
        h_t_proj = self.projection(h_t)  # (batch, 128)

        # Classificação
        logits = self.classifier(h_t_proj)  # (batch, 1)
        return logits.squeeze(1)

model = BAHTextModel(freeze_bert=True)
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9561.78it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total params: 109,588,993
Trainable params: 106,753


In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)

def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            logits = model(input_ids, attention_mask)
            preds = (torch.sigmoid(logits) > 0.5).float()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    return macro_f1

# Treino
num_epochs = 20
best_val_f1 = 0

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    val_f1 = evaluate(model, val_loader)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), 'best_text_model.pt')

    print(f"Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f} | Val Macro F1: {val_f1:.4f} | Best: {best_val_f1:.4f}")

# Avaliar no test com melhor modelo
model.load_state_dict(torch.load('best_text_model.pt'))
test_f1 = evaluate(model, test_loader)
print(f"\nTest Macro F1: {test_f1:.4f}")

Epoch 1/20 | Loss: 0.6999 | Val Macro F1: 0.2989 | Best: 0.2989
Epoch 2/20 | Loss: 0.6910 | Val Macro F1: 0.4403 | Best: 0.4403
Epoch 3/20 | Loss: 0.6903 | Val Macro F1: 0.5003 | Best: 0.5003
Epoch 4/20 | Loss: 0.6891 | Val Macro F1: 0.5163 | Best: 0.5163
Epoch 5/20 | Loss: 0.6830 | Val Macro F1: 0.4325 | Best: 0.5163
Epoch 6/20 | Loss: 0.6831 | Val Macro F1: 0.4858 | Best: 0.5163
Epoch 7/20 | Loss: 0.6758 | Val Macro F1: 0.5542 | Best: 0.5542
Epoch 8/20 | Loss: 0.6723 | Val Macro F1: 0.5319 | Best: 0.5542
Epoch 9/20 | Loss: 0.6762 | Val Macro F1: 0.4462 | Best: 0.5542
Epoch 10/20 | Loss: 0.6667 | Val Macro F1: 0.4883 | Best: 0.5542
Epoch 11/20 | Loss: 0.6700 | Val Macro F1: 0.4883 | Best: 0.5542
Epoch 12/20 | Loss: 0.6611 | Val Macro F1: 0.5506 | Best: 0.5542
Epoch 13/20 | Loss: 0.6564 | Val Macro F1: 0.5258 | Best: 0.5542
Epoch 14/20 | Loss: 0.6594 | Val Macro F1: 0.5328 | Best: 0.5542
Epoch 15/20 | Loss: 0.6571 | Val Macro F1: 0.5542 | Best: 0.5542
Epoch 16/20 | Loss: 0.6520 | Val M